# 01 — Cement Unit Break Analysis
Visualises the structural break in `Consommation_ciment` (§3.1).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
from lamiaty.utils.logging import setup_logging
from lamiaty.config import load_settings
from lamiaty.data.loader import load_base_btp, COL_CEMENT
from lamiaty.data.corrections import fix_cement_unit_break
from lamiaty.visualization.diagnostics import plot_series_with_break
import warnings

setup_logging()
settings = load_settings("../configs", project_root="..")
df_raw = load_base_btp(settings.paths.base_btp_path)

## Raw series — break visible

In [ ]:
fig = plot_series_with_break(
    df_raw[COL_CEMENT],
    break_date=settings.corrections.cement_break.break_date,
    title="Consommation ciment — RAW (structural break visible at ~×759)"
)
fig.savefig("../docs/cement_raw_break.png", dpi=150, bbox_inches="tight")

## Corrected series

In [ ]:
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    corrected = fix_cement_unit_break(
        df_raw[COL_CEMENT],
        break_date=settings.corrections.cement_break.break_date,
        factor=settings.corrections.cement_break.correction_factor,
        confirmed_by=settings.corrections.cement_break.confirmed_by,
    )
    if w:
        print("⚠️ Warning:", str(w[0].message))

fig = plot_series_with_break(
    corrected,
    break_date=settings.corrections.cement_break.break_date,
    title=f"Consommation ciment — CORRECTED (factor={settings.corrections.cement_break.correction_factor})"
)
fig.savefig("../docs/cement_corrected.png", dpi=150, bbox_inches="tight")

## Ratio check around break date

In [ ]:
import pandas as pd
break_ts = pd.Timestamp(settings.corrections.cement_break.break_date)
window = df_raw[COL_CEMENT].loc[
    (df_raw.index >= break_ts - pd.DateOffset(months=3)) &
    (df_raw.index <= break_ts + pd.DateOffset(months=3))
]
print("Values around break date:")
print(window.to_string())
print(f"\nPost-break / Pre-break ratio: {window.iloc[-1] / window.iloc[0]:.0f}")
print("Expected ~759 — confirm with APC before setting confirmed_by in corrections.yaml")